In [2]:
import pandas as pd
import numpy as np

def reduce_mem_usage(df):
    start_mem = df.memory_usage().sum() / 1024**2
    print(f'Initial memory: {start_mem:.2f} MB')
    
    for col in df.columns:
        col_type = df[col].dtype
        
        #testing if it is a number
        if pd.api.types.is_numeric_dtype(col_type):
            c_min = df[col].min()
            c_max = df[col].max()
            
            #if it is a integer
            if str(col_type).startswith('int'):
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
            #if it is a float
            else:
                if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
                    df[col] = df[col].astype(np.float16)
                elif c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)
                    
        #if it's text, we convert it to 'category' to save RAM
        elif str(col_type) == 'object' or str(col_type) == 'string':
            if col != 'date':
                df[col] = df[col].astype('category')
                
    end_mem = df.memory_usage().sum() / 1024**2
    return df

In [3]:
#load
df_sales = pd.read_csv('data/sales_train_evaluation.csv')
df_calendar = pd.read_csv('data/calendar.csv')
df_prices = pd.read_csv('data/sell_prices.csv')

#transformation
id_vars = ['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id']
df_ts = pd.melt(df_sales, 
                id_vars=id_vars, 
                var_name='d', 
                value_name='sales')

#downcasting
df_ts = reduce_mem_usage(df_ts)
df_calendar = reduce_mem_usage(df_calendar)
df_prices = reduce_mem_usage(df_prices)

#cross with date
cols_calendario = ['d', 'date', 'wm_yr_wk', 'event_name_1', 'snap_CA', 'snap_TX', 'snap_WI']
df_final = pd.merge(df_ts, df_calendar[cols_calendario], on='d', how='left')
df_final['date'] = pd.to_datetime(df_final['date'])

#cross with prices
df_final = pd.merge(df_final, df_prices, 
                    on=['store_id', 'item_id', 'wm_yr_wk'], 
                    how='left')

#free RAM
del df_sales, df_ts, df_calendar, df_prices
import gc
gc.collect()

df_final.info()

Initial memory: 3612.13 MB
Initial memory: 0.21 MB
Initial memory: 208.77 MB
<class 'pandas.DataFrame'>
RangeIndex: 59181090 entries, 0 to 59181089
Data columns (total 15 columns):
 #   Column        Dtype         
---  ------        -----         
 0   id            str           
 1   item_id       str           
 2   dept_id       str           
 3   cat_id        str           
 4   store_id      str           
 5   state_id      str           
 6   d             str           
 7   sales         int16         
 8   date          datetime64[us]
 9   wm_yr_wk      int16         
 10  event_name_1  str           
 11  snap_CA       int8          
 12  snap_TX       int8          
 13  snap_WI       int8          
 14  sell_price    float16       
dtypes: datetime64[us](1), float16(1), int16(2), int8(3), str(8)
memory usage: 4.5 GB


In [4]:
#VALIDATION STRATEGY & DATA LEAKAGE PREVENTION

#isolate store CA_1 and department FOODS_3 
df_ca1_foods3 = df_final[(df_final['store_id'] == 'CA_1') & (df_final['dept_id'] == 'FOODS_3')]

#the top 10 historically best-selling products from this subset
top_10_items = df_ca1_foods3.groupby('item_id')['sales'].sum().nlargest(10).index.tolist()
print(f"The top 10 products to forecast are: {top_10_items}")

df_models = df_ca1_foods3[df_ca1_foods3['item_id'].isin(top_10_items)].copy()

#extract the day number (e.g., 'd_1' -> 1) to allow chronological splitting
df_models['day_num'] = df_models['d'].str.replace('d_', '').astype(int)

train = df_models[df_models['day_num'] <= 1885].copy()
val   = df_models[(df_models['day_num'] >= 1886) & (df_models['day_num'] <= 1913)].copy()
test  = df_models[df_models['day_num'] >= 1914].copy()

print(f"TRAIN Set      (Days 1-1885)   : {len(train):,} rows")
print(f"VALIDATION Set (Days 1886-1913): {len(val):,} rows")
print(f"TEST Set       (Days 1914-1941): {len(test):,} rows")


The top 10 products to forecast are: ['FOODS_3_090', 'FOODS_3_586', 'FOODS_3_252', 'FOODS_3_120', 'FOODS_3_714', 'FOODS_3_587', 'FOODS_3_808', 'FOODS_3_080', 'FOODS_3_555', 'FOODS_3_541']
TRAIN Set      (Days 1-1885)   : 18,850 rows
VALIDATION Set (Days 1886-1913): 280 rows
TEST Set       (Days 1914-1941): 280 rows


In [5]:
from sklearn.metrics import mean_squared_error, mean_absolute_error

#ALGORITHMIC BASELINES

#validation set
results_val = val[['item_id', 'd', 'day_num', 'date', 'sales']].copy()
results_val.rename(columns={'sales': 'y_true'}, inplace=True)

#MODEL 1: NAÏVE FORECAST
#Capturamos las ventas del último día de entrenamiento (Día 1885)
last_day_train = train[train['day_num'] == 1885][['item_id', 'sales']]
last_day_train.rename(columns={'sales': 'pred_naive'}, inplace=True)

# Proyectamos ese único valor hacia adelante para los 28 días de validación
results_val = pd.merge(results_val, last_day_train, on='item_id', how='left')



#MODELO 2: SEASONAL NAÏVE FORECAST

print("Training Seasonal Naïve Model (Predicting last month's sequence...)")
# Capturamos la secuencia exacta de los últimos 28 días de entrenamiento
last_28_train = train[(train['day_num'] >= 1858) & (train['day_num'] <= 1885)][['item_id', 'day_num', 'sales']]

# Desplazamos la línea temporal matemáticamente sumando 28 días
last_28_train['day_num_val'] = last_28_train['day_num'] + 28
last_28_train.rename(columns={'sales': 'pred_snaive'}, inplace=True)

# Cruzamos los datos alineando las fechas proyectadas con las reales
results_val = pd.merge(results_val, last_28_train[['item_id', 'day_num_val', 'pred_snaive']],
                       left_on=['item_id', 'day_num'],
                       right_on=['item_id', 'day_num_val'],
                       how='left').drop(columns=['day_num_val'])


# ---------------------------------------------------------
# EVALUATION METRICS (RMSE & MAE)
# ---------------------------------------------------------
def calculate_metrics(y_true, y_pred, model_name):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    print(f"Results for {model_name}:")
    print(f"  --> RMSE: {rmse:.4f}")
    print(f"  --> MAE:  {mae:.4f}\n")

print("\n" + "="*50)
print("BASELINE PERFORMANCE (VALIDATION SET: 28 DAYS)")
print("="*50)
calculate_metrics(results_val['y_true'], results_val['pred_naive'], "Naïve Approach")
calculate_metrics(results_val['y_true'], results_val['pred_snaive'], "Seasonal Naïve Approach")

# Opcional: Mostrar una muestra visual de cómo ha quedado la tabla de predicciones
# display(results_val.head(10))

Training Seasonal Naïve Model (Predicting last month's sequence...)

BASELINE PERFORMANCE (VALIDATION SET: 28 DAYS)
Results for Naïve Approach:
  --> RMSE: 29.1298
  --> MAE:  19.8750

Results for Seasonal Naïve Approach:
  --> RMSE: 18.5143
  --> MAE:  10.3857



In [6]:
from statsmodels.tsa.statespace.sarimax import SARIMAX
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error
import warnings
warnings.filterwarnings('ignore') # Ocultamos los warnings matemáticos para no ensuciar la pantalla

print("--- PHASE 3: UNIVARIATE STATISTICAL MODELING (SARIMA) ---")
print("Iniciando entrenamiento de 10 modelos SARIMA individuales...")
print("⚠️ Aviso: Este proceso es computacionalmente pesado. Puede tardar varios minutos.\n")

# Lista para guardar las predicciones de SARIMA
predicciones_sarima = []

# Parámetros base extraídos del análisis ACF/PACF y estacionalidad semanal
p, d, q = 1, 1, 1
P, D, Q, m = 1, 0, 1, 7

# Entrenamos un modelo por cada producto
for i, producto in enumerate(top_10_items, 1):
    print(f"[{i}/10] Entrenando SARIMA para {producto}...")
    
    # Aislar datos del producto
    df_train_prod = train[train['item_id'] == producto].sort_values('day_num')
    y_train = df_train_prod['sales'].values
    
    # Definir el modelo SARIMA (Sin variables exógenas en esta fase)
    modelo_sarima = SARIMAX(y_train, 
                            order=(p, d, q), 
                            seasonal_order=(P, D, Q, m),
                            enforce_stationarity=False, 
                            enforce_invertibility=False)
    
    # Ajustar el modelo (Aquí es donde el procesador sufre)
    modelo_ajustado = modelo_sarima.fit(disp=False)
    
    # Predecir los 28 días de validación
    prediccion_28 = modelo_ajustado.forecast(steps=28)
    
    # Guardar resultados
    df_prod_preds = pd.DataFrame({
        'item_id': producto,
        'day_num': range(1886, 1914),
        'pred_sarima': prediccion_28
    })
    
    predicciones_sarima.append(df_prod_preds)

print("\n¡Entrenamiento completado!")

# Consolidamos las predicciones
df_preds_fase3_sarima = pd.concat(predicciones_sarima, ignore_index=True)

# Cruzamos con nuestra tabla maestra de resultados de validación (results_val)
results_val = pd.merge(results_val, df_preds_fase3_sarima, on=['item_id', 'day_num'], how='left')

# Evitar predicciones negativas por fluctuaciones matemáticas
results_val['pred_sarima'] = results_val['pred_sarima'].clip(lower=0)

print("\n" + "="*50)
print("PHASE 3 PERFORMANCE (SARIMA Univariante)")
print("="*50)

# Reutilizamos tu función de cálculo de métricas
def calculate_metrics(y_true, y_pred, model_name):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    print(f"Results for {model_name}:")
    print(f"  --> RMSE: {rmse:.4f}")
    print(f"  --> MAE:  {mae:.4f}\n")

calculate_metrics(results_val['y_true'], results_val['pred_sarima'], "SARIMA (1,1,1)(1,0,1,7)")

--- PHASE 3: UNIVARIATE STATISTICAL MODELING (SARIMA) ---
Iniciando entrenamiento de 10 modelos SARIMA individuales...
⚠️ Aviso: Este proceso es computacionalmente pesado. Puede tardar varios minutos.

[1/10] Entrenando SARIMA para FOODS_3_090...
[2/10] Entrenando SARIMA para FOODS_3_586...
[3/10] Entrenando SARIMA para FOODS_3_252...
[4/10] Entrenando SARIMA para FOODS_3_120...
[5/10] Entrenando SARIMA para FOODS_3_714...
[6/10] Entrenando SARIMA para FOODS_3_587...
[7/10] Entrenando SARIMA para FOODS_3_808...
[8/10] Entrenando SARIMA para FOODS_3_080...
[9/10] Entrenando SARIMA para FOODS_3_555...
[10/10] Entrenando SARIMA para FOODS_3_541...

¡Entrenamiento completado!

PHASE 3 PERFORMANCE (SARIMA Univariante)
Results for SARIMA (1,1,1)(1,0,1,7):
  --> RMSE: 14.6747
  --> MAE:  9.1084



In [11]:
import pandas as pd
import numpy as np
from statsmodels.tsa.statespace.sarimax import SARIMAX
import warnings
warnings.filterwarnings('ignore')

print("--- PHASE 4: MULTIVARIATE STATISTICAL MODELING (SARIMAX) ---")
print("Inyectando el mundo real: Ayudas SNAP, Eventos del Calendario y Precios...\n")

# ---------------------------------------------------------
# 1. PREPARACIÓN DE LAS VARIABLES EXÓGENAS (Data Wrangling)
# ---------------------------------------------------------
def preparar_exogenas(df):
    df_exog = df.copy()
    
    # A. Ayudas SNAP
    
    # B. Eventos del calendario
    df_exog['event_flag'] = df_exog['event_name_1'].notna().astype(int)
    
    # C. Precios (SOLUCIÓN AL ERROR DE PANDAS)
    df_exog['sell_price'] = df_exog['sell_price'].astype('float32')
    df_exog['sell_price'] = df_exog.groupby('item_id')['sell_price'].ffill()
    df_exog['sell_price'] = df_exog.groupby('item_id')['sell_price'].bfill()
    df_exog['sell_price'] = df_exog['sell_price'].fillna(0)
    
    return df_exog

train_exog = preparar_exogenas(train)
val_exog = preparar_exogenas(val)

columnas_exogenas = ['snap_CA', 'event_flag', 'sell_price']

# ---------------------------------------------------------
# 2. ENTRENAMIENTO DEL MODELO SARIMAX MULTIVARIANTE
# ---------------------------------------------------------
predicciones_sarimax = []

p, d, q = 1, 1, 1
P, D, Q, m = 1, 0, 1, 7

for i, producto in enumerate(top_10_items, 1):
    print(f"[{i}/10] Entrenando SARIMAX (Context-Aware) para {producto}...")
    
    df_train_prod = train_exog[train_exog['item_id'] == producto].sort_values('day_num')
    y_train = df_train_prod['sales'].values
    X_train = df_train_prod[columnas_exogenas].values
    
    df_val_prod = val_exog[val_exog['item_id'] == producto].sort_values('day_num')
    X_val = df_val_prod[columnas_exogenas].values
    
    modelo_sarimax = SARIMAX(y_train, 
                             exog=X_train, 
                             order=(p, d, q), 
                             seasonal_order=(P, D, Q, m),
                             enforce_stationarity=False, 
                             enforce_invertibility=False)
    
    modelo_ajustado = modelo_sarimax.fit(disp=False)
    
    # ---> SOLUCIÓN AQUÍ: Quitamos el .values porque ya es un numpy array <---
    prediccion_28 = modelo_ajustado.forecast(steps=28, exog=X_val)
    
    df_prod_preds = pd.DataFrame({
        'item_id': producto,
        'day_num': range(1886, 1914),
        'pred_sarimax_full': prediccion_28
    })
    
    predicciones_sarimax.append(df_prod_preds)

print("\n¡Entrenamiento Multivariante completado!")

# ---------------------------------------------------------
# 3. EVALUACIÓN Y COMPARACIÓN
# ---------------------------------------------------------
df_preds_fase4 = pd.concat(predicciones_sarimax, ignore_index=True)

# Si la columna ya existe de un intento anterior, la borramos para que cruce limpio
if 'pred_sarimax_full' in results_val.columns:
    results_val.drop(columns=['pred_sarimax_full'], inplace=True)

results_val = pd.merge(results_val, df_preds_fase4, on=['item_id', 'day_num'], how='left')

# Evitamos predicciones negativas
results_val['pred_sarimax_full'] = results_val['pred_sarimax_full'].clip(lower=0)

print("\n" + "="*50)
print("PHASE 4 PERFORMANCE (SARIMAX FULL)")
print("="*50)
calculate_metrics(results_val['y_true'], results_val['pred_sarimax_full'], "SARIMAX (SNAP + Eventos + Precios)")

--- PHASE 4: MULTIVARIATE STATISTICAL MODELING (SARIMAX) ---
Inyectando el mundo real: Ayudas SNAP, Eventos del Calendario y Precios...

[1/10] Entrenando SARIMAX (Context-Aware) para FOODS_3_090...
[2/10] Entrenando SARIMAX (Context-Aware) para FOODS_3_586...
[3/10] Entrenando SARIMAX (Context-Aware) para FOODS_3_252...
[4/10] Entrenando SARIMAX (Context-Aware) para FOODS_3_120...
[5/10] Entrenando SARIMAX (Context-Aware) para FOODS_3_714...
[6/10] Entrenando SARIMAX (Context-Aware) para FOODS_3_587...
[7/10] Entrenando SARIMAX (Context-Aware) para FOODS_3_808...
[8/10] Entrenando SARIMAX (Context-Aware) para FOODS_3_080...
[9/10] Entrenando SARIMAX (Context-Aware) para FOODS_3_555...
[10/10] Entrenando SARIMAX (Context-Aware) para FOODS_3_541...

¡Entrenamiento Multivariante completado!

PHASE 4 PERFORMANCE (SARIMAX FULL)
Results for SARIMAX (SNAP + Eventos + Precios):
  --> RMSE: 15.5783
  --> MAE:  9.8465



In [ ]:
import pandas as pd
import numpy as np
from statsmodels.tsa.statespace.sarimax import SARIMAX
import warnings
warnings.filterwarnings('ignore')

print("--- PHASE 4 (OPTIMIZADA): MULTIVARIATE STATISTICAL MODELING (SARIMAX) ---")
print("Filtrando el ruido: Usando ÚNICAMENTE 'snap_CA' como variable exógena...\n")

# ---------------------------------------------------------
# 1. AISLAR LA VARIABLE CLAVE
# ---------------------------------------------------------
# Como vimos que los Eventos y Precios confunden al modelo,
# vamos a usar solo la variable que demostró impacto real: SNAP.
columnas_exogenas = ['snap_CA']

# ---------------------------------------------------------
# 2. ENTRENAMIENTO DEL MODELO SARIMAX OPTIMIZADO
# ---------------------------------------------------------
predicciones_sarimax_opt = []

# Mantenemos los parámetros matemáticos que funcionaron en la Fase 3
p, d, q = 1, 1, 1
P, D, Q, m = 1, 0, 1, 7

for i, producto in enumerate(top_10_items, 1):
    print(f"[{i}/10] Entrenando SARIMAX Optimizado (Solo SNAP) para {producto}...")
    
    # Aislar datos de entrenamiento (Y = Ventas, X = SNAP)
    df_train_prod = train[train['item_id'] == producto].sort_values('day_num')
    y_train = df_train_prod['sales'].values
    X_train = df_train_prod[columnas_exogenas].values
    
    # Aislar datos de validación (El calendario futuro de SNAP)
    df_val_prod = val[val['item_id'] == producto].sort_values('day_num')
    X_val = df_val_prod[columnas_exogenas].values
    
    # Definir modelo inyectando la matriz 'exog'
    modelo_sarimax = SARIMAX(y_train, 
                             exog=X_train, 
                             order=(p, d, q), 
                             seasonal_order=(P, D, Q, m),
                             enforce_stationarity=False, 
                             enforce_invertibility=False)
    
    # Ajustar modelo
    modelo_ajustado = modelo_sarimax.fit(disp=False)
    
    # Predecir de forma segura
    prediccion_28 = modelo_ajustado.forecast(steps=28, exog=X_val)
    
    # Guardar resultados
    df_prod_preds = pd.DataFrame({
        'item_id': producto,
        'day_num': range(1886, 1914),
        'pred_sarimax_opt': prediccion_28
    })
    
    predicciones_sarimax_opt.append(df_prod_preds)

print("\n¡Entrenamiento Multivariante Optimizado completado!")

# ---------------------------------------------------------
# 3. EVALUACIÓN Y COMPARACIÓN (CON ESCUDO ANTI-ERRORES)
# ---------------------------------------------------------
df_preds_fase4_opt = pd.concat(predicciones_sarimax_opt, ignore_index=True)

# Escudo anti-KeyError: Borramos la columna si existe de un intento anterior
if 'pred_sarimax_opt' in results_val.columns:
    results_val.drop(columns=['pred_sarimax_opt'], inplace=True)

# Cruzamos los nuevos resultados
results_val = pd.merge(results_val, df_preds_fase4_opt, on=['item_id', 'day_num'], how='left')

# Evitamos predicciones negativas (recorte a 0)
results_val['pred_sarimax_opt'] = results_val['pred_sarimax_opt'].clip(lower=0)

print("\n" + "="*50)
print("PHASE 4 PERFORMANCE (SARIMAX OPTIMIZADO)")
print("="*50)

# Calculamos las métricas
calculate_metrics(results_val['y_true'], results_val['pred_sarimax_opt'], "SARIMAX (Solo SNAP)")

--- PHASE 4 (OPTIMIZADA): MULTIVARIATE STATISTICAL MODELING (SARIMAX) ---
Filtrando el ruido: Usando ÚNICAMENTE 'snap_CA' como variable exógena...

[1/10] Entrenando SARIMAX Optimizado (Solo SNAP) para FOODS_3_090...
[2/10] Entrenando SARIMAX Optimizado (Solo SNAP) para FOODS_3_586...
[3/10] Entrenando SARIMAX Optimizado (Solo SNAP) para FOODS_3_252...
[4/10] Entrenando SARIMAX Optimizado (Solo SNAP) para FOODS_3_120...
[5/10] Entrenando SARIMAX Optimizado (Solo SNAP) para FOODS_3_714...
[6/10] Entrenando SARIMAX Optimizado (Solo SNAP) para FOODS_3_587...
[7/10] Entrenando SARIMAX Optimizado (Solo SNAP) para FOODS_3_808...
[8/10] Entrenando SARIMAX Optimizado (Solo SNAP) para FOODS_3_080...
[9/10] Entrenando SARIMAX Optimizado (Solo SNAP) para FOODS_3_555...
[10/10] Entrenando SARIMAX Optimizado (Solo SNAP) para FOODS_3_541...

¡Entrenamiento Multivariante Optimizado completado!

PHASE 4 PERFORMANCE (SARIMAX OPTIMIZADO)
Results for SARIMAX (Solo SNAP):
  --> RMSE: 14.7133
  --> MAE:  9

In [13]:
import pandas as pd
import numpy as np
from statsmodels.tsa.statespace.sarimax import SARIMAX
import warnings
warnings.filterwarnings('ignore')

print("--- PHASE 4 (ULTIMATE): SARIMAX CON DIFERENCIACIÓN ESTACIONAL ---")
print("Aplicando D=1 para forzar la comparación 'semana a semana' junto con SNAP...\n")

columnas_exogenas = ['snap_CA']
predicciones_sarimax_max = []

# EL CAMBIO MÁGICO: Pasamos D de 0 a 1. 
# Esto fuerza la Diferenciación Estacional (restar el valor de hace 7 días)
p, d, q = 1, 1, 1
P, D, Q, m = 1, 1, 1, 7 

for i, producto in enumerate(top_10_items, 1):
    print(f"[{i}/10] Entrenando SARIMAX Ultimate para {producto}...")
    
    df_train_prod = train[train['item_id'] == producto].sort_values('day_num')
    y_train = df_train_prod['sales'].values
    X_train = df_train_prod[columnas_exogenas].values
    
    df_val_prod = val[val['item_id'] == producto].sort_values('day_num')
    X_val = df_val_prod[columnas_exogenas].values
    
    # Modelo con los nuevos parámetros estacionales
    modelo_sarimax = SARIMAX(y_train, 
                             exog=X_train, 
                             order=(p, d, q), 
                             seasonal_order=(P, D, Q, m),
                             enforce_stationarity=False, 
                             enforce_invertibility=False)
    
    modelo_ajustado = modelo_sarimax.fit(disp=False)
    prediccion_28 = modelo_ajustado.forecast(steps=28, exog=X_val)
    
    df_prod_preds = pd.DataFrame({
        'item_id': producto,
        'day_num': range(1886, 1914),
        'pred_sarimax_max': prediccion_28
    })
    
    predicciones_sarimax_max.append(df_prod_preds)

print("\n¡Entrenamiento Ultimate completado!")

# ---------------------------------------------------------
# EVALUACIÓN Y COMPARACIÓN
# ---------------------------------------------------------
df_preds_fase4_max = pd.concat(predicciones_sarimax_max, ignore_index=True)

# Limpieza por si lo ejecutas varias veces
if 'pred_sarimax_max' in results_val.columns:
    results_val.drop(columns=['pred_sarimax_max'], inplace=True)

results_val = pd.merge(results_val, df_preds_fase4_max, on=['item_id', 'day_num'], how='left')
results_val['pred_sarimax_max'] = results_val['pred_sarimax_max'].clip(lower=0)

print("\n" + "="*50)
print("PHASE 4 PERFORMANCE (SARIMAX ULTIMATE)")
print("="*50)
calculate_metrics(results_val['y_true'], results_val['pred_sarimax_max'], "SARIMAX (SNAP + D=1)")

--- PHASE 4 (ULTIMATE): SARIMAX CON DIFERENCIACIÓN ESTACIONAL ---
Aplicando D=1 para forzar la comparación 'semana a semana' junto con SNAP...

[1/10] Entrenando SARIMAX Ultimate para FOODS_3_090...
[2/10] Entrenando SARIMAX Ultimate para FOODS_3_586...
[3/10] Entrenando SARIMAX Ultimate para FOODS_3_252...
[4/10] Entrenando SARIMAX Ultimate para FOODS_3_120...
[5/10] Entrenando SARIMAX Ultimate para FOODS_3_714...
[6/10] Entrenando SARIMAX Ultimate para FOODS_3_587...
[7/10] Entrenando SARIMAX Ultimate para FOODS_3_808...
[8/10] Entrenando SARIMAX Ultimate para FOODS_3_080...
[9/10] Entrenando SARIMAX Ultimate para FOODS_3_555...
[10/10] Entrenando SARIMAX Ultimate para FOODS_3_541...

¡Entrenamiento Ultimate completado!

PHASE 4 PERFORMANCE (SARIMAX ULTIMATE)
Results for SARIMAX (SNAP + D=1):
  --> RMSE: 13.6673
  --> MAE:  8.5745



In [14]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
import warnings
warnings.filterwarnings('ignore')

print("=================================================================")
print("🧠 INICIANDO FASE 5: DEEP LEARNING & CROSS-LEARNING (LSTM)")
print("=================================================================\n")

# ---------------------------------------------------------
# 1. TRANSFORMACIÓN A MATRIZ GLOBAL (Cross-Learning)
# ---------------------------------------------------------
print("1. Reestructurando datos para visión global...")
# Pivotamos para tener los 10 productos en columnas (1 fila = 1 día)
df_pivot_train = train.pivot(index='day_num', columns='item_id', values='sales').fillna(0)

# Añadimos nuestra variable exógena estrella (SNAP) al mismo DataFrame
snap_train = train.drop_duplicates('day_num').set_index('day_num')['snap_CA']
df_pivot_train['snap_CA'] = snap_train

# Guardamos el orden de las columnas de los productos para luego
item_columns = df_pivot_train.columns.drop('snap_CA')

# ---------------------------------------------------------
# 2. ESCALADO DE DATOS (MinMaxScaler)
# ---------------------------------------------------------
print("2. Escalando datos entre 0 y 1 para la Red Neuronal...")
scaler = MinMaxScaler()
# Escalamos toda la matriz de entrenamiento
scaled_train = scaler.fit_transform(df_pivot_train)

# ---------------------------------------------------------
# 3. CREACIÓN DE VENTANAS TEMPORALES (Sliding Windows)
# ---------------------------------------------------------
print("3. Generando tensores 3D (Ventanas de 28 días)...")
WINDOW_SIZE = 28 # Miramos 28 días hacia atrás...
# ...para predecir los 10 productos del día siguiente

X_train_lstm = []
y_train_lstm = []

for i in range(WINDOW_SIZE, len(scaled_train)):
    # X: Bloque de 28 días con las 11 variables (10 items + 1 SNAP)
    X_train_lstm.append(scaled_train[i - WINDOW_SIZE : i])
    # Y: Solo queremos predecir las ventas (las primeras 10 columnas), no el SNAP
    y_train_lstm.append(scaled_train[i, :-1]) 

X_train_lstm = np.array(X_train_lstm)
y_train_lstm = np.array(y_train_lstm)

print(f"Forma del Tensor X (Entrada): {X_train_lstm.shape} -> (Muestras, Días, Variables)")
print(f"Forma del Tensor Y (Salida) : {y_train_lstm.shape} -> (Muestras, Productos a predecir)\n")

# ---------------------------------------------------------
# 4. ARQUITECTURA DE LA RED NEURONAL LSTM
# ---------------------------------------------------------
print("4. Construyendo la arquitectura LSTM...")

# Semilla para que los resultados sean reproducibles
tf.random.set_seed(42)

modelo_lstm = Sequential([
    # Capa LSTM que procesa la secuencia temporal
    LSTM(64, activation='relu', input_shape=(X_train_lstm.shape[1], X_train_lstm.shape[2])),
    # Capa Dropout para apagar neuronas aleatorias y evitar el Overfitting
    Dropout(0.2),
    # Capa de salida: 10 neuronas (una para cada producto)
    Dense(10) 
])

modelo_lstm.compile(optimizer='adam', loss='mse')
modelo_lstm.summary()

# ---------------------------------------------------------
# 5. ENTRENAMIENTO GLOBAL
# ---------------------------------------------------------
print("\n5. Entrenando el cerebro global... (Esto puede tardar un poco)")
historia = modelo_lstm.fit(X_train_lstm, y_train_lstm, 
                           epochs=30,          # Veces que ve el dataset completo
                           batch_size=32,      # Muestras que procesa de golpe
                           verbose=1,
                           shuffle=False)      # NO mezclar en series temporales

print("\n¡Entrenamiento de la Red Neuronal completado!")

ModuleNotFoundError: No module named 'tensorflow'